# Lab 2 — Fuentes de Datos Publicos: Calidad y Limitaciones
### TC4036 · Ciencia de Datos para Politicas Publicas · Semana 2

---

**Objetivo:** Desarrollar el habito de hacer preguntas criticas sobre cualquier
dataset *antes* de analizarlo. El codigo de hoy es minimo — el trabajo es conceptual.

**Estructura:**
- **Parte 1 (~25 min):** Mapa de fuentes de datos publicos + criterios de evaluacion
- **Parte 2 (~30 min):** Explorar el dataset CONEVAL — ¿que tiene? ¿que le falta? ¿a quien le falta?
- **Parte 3 (~25 min):** Tu proyecto — evalua tu propia fuente de datos

---
> Antes de ejecutar este notebook, sube el archivo  
> `Anexo_estad_stico_entidades_2022.xlsx` a tu sesion de Colab  
> o ponlo en el mismo directorio que este notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Librerias cargadas.')

---
## Parte 1 — Mapa de fuentes de datos publicos

Un analista de datos para politicas publicas necesita saber donde viven los datos
relevantes para su area. Este catalogo es un punto de partida — no es exhaustivo.

In [ ]:
# Un catalogo de fuentes no es codigo analitico — es un mapa de referencia.
# En proyectos reales, construir algo asi es el primer paso.

fuentes = pd.DataFrame([
    # Mexico
    dict(fuente='CONEVAL',             tema='Pobreza y bienestar',
         cobertura='Estatal/municipal',frecuencia='Bianual',  formato='XLSX/CSV',
         url='coneval.org.mx'),
    dict(fuente='INEGI — ENIGH',       tema='Ingreso y gasto de hogares',
         cobertura='Estatal',          frecuencia='Bianual',  formato='CSV',
         url='inegi.org.mx/programas/enigh'),
    dict(fuente='INEGI — Censo',       tema='Demografia y vivienda',
         cobertura='Municipal',        frecuencia='Decenal',  formato='CSV/SHP',
         url='inegi.org.mx/programas/ccpv/2020'),
    dict(fuente='datos.gob.mx',        tema='Portal de datos abiertos',
         cobertura='Variable',         frecuencia='Variable', formato='CSV/JSON',
         url='datos.gob.mx'),
    dict(fuente='IMSS datos abiertos', tema='Empleo formal registrado',
         cobertura='Municipal',        frecuencia='Mensual',  formato='CSV',
         url='datos.imss.gob.mx'),
    # Internacional
    dict(fuente='Our World in Data',   tema='Clima, salud, economia, bienestar',
         cobertura='Pais',             frecuencia='Anual',    formato='CSV',
         url='ourworldindata.org'),
    dict(fuente='World Bank — WDI',    tema='Desarrollo economico y social',
         cobertura='Pais',             frecuencia='Anual',    formato='CSV/API',
         url='data.worldbank.org'),
    dict(fuente='CEPAL CEPALSTAT',     tema='Economia y desarrollo en ALC',
         cobertura='Pais',             frecuencia='Anual',    formato='CSV/API',
         url='estadisticas.cepal.org'),
])

pd.set_option('display.max_colwidth', 40)
fuentes[['fuente', 'tema', 'cobertura', 'frecuencia', 'formato']]

**Discusion rapida:** Mira el catalogo y responde:
- ¿Que fuente usarias para estudiar desnutricion infantil en Guerrero?
- ¿Que fuente usarias para comparar Mexico con Brasil en emision de carbono?
- ¿Hay alguna pregunta de politica publica que ninguna de estas fuentes puede responder?

*Escribe tus respuestas aqui antes de continuar.*

### 1.1 — Antes de usar un dataset: cuatro preguntas

Antes de descargar datos y empezar a correr codigo, un analista responsable
siempre hace estas cuatro preguntas sobre su fuente:

1. **¿Que cubre?** — Unidad de observacion, variables disponibles, periodo
2. **¿Quien esta?** — Que poblacion esta representada y cual no
3. **¿Con que frecuencia?** — ¿Puedo ver cambios en el tiempo o es un snapshot?
4. **¿Bajo que licencia?** — ¿Puedo usar esto en mi trabajo/publicacion?

La funcion `evaluar_fuente()` abajo es una forma de formalizar ese habito.

In [ ]:
# Ejemplo de funciones
def suma(a, b):
    return a + b

def multiplicacion(a, b):
    return a * b

def division(a, b):
    if b != 0:
        return a / b
    else:
        return "Error: División por cero"
    
def saluda(nombre):
    return f"Hola, {nombre}!"

# # Prueba de funciones
# print(suma(3, 5))  # Debería imprimir 8
# print(saluda("Diana"))  # Debería imprimir "Hola, Diana!"

In [ ]:
def suma_explicita(a: float, b: float) -> float:
    """Suma dos números y devuelve el resultado."""
    resultado = a + b
    return resultado

def saluda_explicito(nombre: str) -> str:
    """Genera un saludo personalizado."""
    saludo = f"Hola, {nombre}!"
    return saludo

# # Prueba de función con anotaciones de tipo

In [ ]:
def evaluar_fuente(nombre, preguntas_posibles, preguntas_imposibles,
                   poblaciones_ausentes, notas_licencia):
    """Tarjeta de evaluacion rapida para una fuente de datos."""
    sep = '-' * 62
    print(f'\n{sep}')
    print(f'  EVALUACION: {nombre}')
    print(sep)
    print('  SI puede responder:')
    for p in preguntas_posibles:
        print(f'    + {p}')
    print('\n  NO puede responder:')
    for p in preguntas_imposibles:
        print(f'    - {p}')
    print('\n  Poblaciones sub-representadas o ausentes:')
    for p in poblaciones_ausentes:
        print(f'    ? {p}')
    print(f'\n  Licencia: {notas_licencia}')
    print(sep)


evaluar_fuente(
    nombre='INEGI — ENIGH 2022',
    preguntas_posibles=[
        'Ingreso promedio de hogares por estado',
        'Proporcion del gasto en alimentacion vs. educacion',
        'Acceso a servicios basicos por nivel socioeconomico',
    ],
    preguntas_imposibles=[
        'Cambios mes a mes (es una encuesta bianual)',
        'Situacion de hogares en asentamientos irregulares',
        'Ingreso de migrantes en transito o trabajadores estacionales',
        'Que causo los cambios observados (describe, no explica)',
    ],
    poblaciones_ausentes=[
        'Personas sin domicilio fijo',
        'Poblacion en instituciones (carceles, hospitales, albergues)',
        'Comunidades de menos de 2,500 hab. (muestra insuficiente)',
    ],
    notas_licencia='Libre para investigacion y analisis. Citar: INEGI, ENIGH 2022.'
)

**Ejercicio:** Completa una tarjeta para otra fuente del catalogo.
Elige una que sea relevante para el tema de tu proyecto.

In [ ]:
# Completa con la fuente que elegiste
# evaluar_fuente(
#     nombre='NOMBRE DE TU FUENTE',
#     preguntas_posibles=[...],
#     preguntas_imposibles=[...],
#     poblaciones_ausentes=[...],
#     notas_licencia='...'
# )

---
## Parte 2 — Explorar el dataset CONEVAL

Tenemos el archivo de medicion multidimensional de la pobreza por entidad
federativa, 2016–2022 (CONEVAL / ENIGH-INEGI).

El archivo viene en formato Excel con una hoja por estado — pensado para impresion,
no para analisis. La siguiente celda lo transforma en un DataFrame tidy.
Solo ejecutala; no te preocupes aun por entender el codigo de parseo.

In [ ]:
# -- CARGA DE DATOS: solo ejecutar -----------------------------------------
# Esta celda parsea el Excel de CONEVAL y produce un DataFrame limpio.
# El parseo de archivos con formato complejo lo veremos en detalle en la Semana 4.

ARCHIVO = 'Anexo_estadistico_entidades_2022.xlsx'  # ajusta la ruta si es necesario

INDICADORES = {
    8:  'pobreza',
    10: 'pobreza_extrema',
    18: 'rezago_educativo',
    19: 'carencia_salud',
    20: 'carencia_seguridad_social',
    21: 'carencia_vivienda',
    22: 'carencia_servicios_basicos',
    23: 'carencia_alimentacion',
}

xl   = pd.ExcelFile(ARCHIVO)
hojas_estados = [s for s in xl.sheet_names
                 if s not in ['Contenido', 'Estados Unidos Mexicanos']]

filas = []
for estado in hojas_estados:
    raw = pd.read_excel(xl, sheet_name=estado, header=None)
    for fila_idx, indicador in INDICADORES.items(): #  returns dict_items[int, str]
        fila = raw.iloc[fila_idx]
        for i, anio in enumerate([2016, 2018, 2020, 2022]): # enumerate is useful for obtaining an indexed list: (0, seq[0]), (1, seq[1]), (2, seq[2])
            filas.append({
                'estado':    estado,
                'indicador': indicador,
                'anio':      anio,
                'porcentaje': float(fila.iloc[3+i]) if pd.notna(fila.iloc[3+i]) else np.nan,
            })

df = pd.DataFrame(filas)
print(f'Dataset listo: {df.shape[0]:,} filas, {df.shape[1]} columnas')
print(f'{df.estado.nunique()} estados, {df.indicador.nunique()} indicadores, anios: {sorted(df.anio.unique())}')

df.iloc[row_indexer, column_indexer], indexación basada en la posición

df.iloc[0] : Selecciona la primera fila.

df.iloc[:, 0] : Selecciona la primera columna.

df.iloc[0:5, 1:3] : Selecciona las primeras 5 filas y las columnas en las posiciones 1 y 2.

### 2.1 — Primera mirada: ¿que tenemos?

Antes de cualquier analisis, siempre tres preguntas:
¿cuantas filas? ¿cuantas columnas? ¿de que tipo es cada variable?

In [ ]:
# ¿Cuantas filas y columnas?
print(f'Filas: {df.shape[0]:,}') # shape: tupla (filas, columnas)
print(f'Columnas: {df.shape[1]}')
print()

# ¿De que tipo es cada columna?
print(df.dtypes)
print()

# ¿Como se ven los primeros registros?
df.head(10)

In [ ]:
# ¿Que estados hay?
print('Estados en el dataset:')
print(sorted(df['estado'].unique()))
print(f'Total: {df.estado.nunique()}')
print()

# ¿Que indicadores hay?
print('Indicadores disponibles:')
for ind in sorted(df['indicador'].unique()):
    print(f'  - {ind}')

In [ ]:
# Estadisticas descriptivas del porcentaje
# Siempre buscar: ¿hay valores imposibles? ¿minimos negativos? ¿maximos >100?

df.groupby('indicador')['porcentaje'].describe().round(1)

**Pausa de lectura — responde antes de continuar:**

Mira los resultados de `.describe()`. Para el indicador `pobreza`:
- ¿Cual es el valor minimo? ¿Maximo? ¿Media?
- ¿Que significa que la media de `carencia_seguridad_social` sea tan alta?
- ¿Algun numero te sorprende? ¿Por que?

*Escribe tus observaciones aqui.*

In [ ]:
# Filtrar por indicador y encontrar el máximo
max_carencia = df[df['indicador'] == 'carencia_alimentacion']['porcentaje'].max()
estado_max_df = df[(df['indicador'] == 'carencia_alimentacion') & (df['porcentaje'] == max_carencia)]
display(estado_max_df)

print(f"Estado con mayor carencia alimentaria: {estado_max_df['estado'].values[0]}")
print(f"Porcentaje: {estado_max_df['porcentaje'].values[0]:.2f}%")
print(f"Anio: {estado_max_df['anio'].values[0]}")

### 2.2 — Valores faltantes

Los faltantes no son solo un problema tecnico — son informacion.
Siempre preguntar: **¿por que falta ese dato?**

In [ ]:
# ¿Cuantos valores faltantes hay?
faltantes = df.isnull().sum()
pct = (faltantes / len(df) * 100).round(1)

print('Valores faltantes por columna:')
for col in df.columns:
    print(f'  {col:15s}  {faltantes[col]:4d}  ({pct[col]:.1f}%)')

print()
if faltantes.sum() == 0:
    print('No hay valores faltantes.')
    print()
    print('Reflexion: ¿que podria significar que un dataset oficial no tenga faltantes?')
    print('  a) Excelente cobertura de medicion')
    print('  b) Los casos dificiles de medir simplemente no se reportan')
    print('  c) Hay imputacion o redondeo no documentado')
    print()
    print('Siempre vale la pena leer la nota metodologica del dataset.')

### 2.3 — Una grafica: pobreza por estado en 2022

Una sola grafica, bien elegida, ya revela mucho.

In [ ]:
# Pobreza por estado en 2022, ordenada de mayor a menor
mask = (df['indicador'] == 'pobreza') & (df['anio'] == 2022)
df_2022 = (
    df[mask]
    .sort_values('porcentaje', ascending=True)
)

# Nombre corto para estados con nombre muy largo
df_2022 = df_2022.copy()
df_2022['estado_corto'] = (
    df_2022['estado']
    .str.replace('Veracruz de Ignacio de la Llave', 'Veracruz')
    .str.replace('Michoacan de Ocampo', 'Michoacan')
    .str.replace('Coahuila de Zaragoza', 'Coahuila')
)

fig, ax = plt.subplots(figsize=(8, 10))
colores = ['#F65905' if v > 50 else '#0891B2' for v in df_2022['porcentaje']]
ax.barh(df_2022['estado_corto'], df_2022['porcentaje'],
        color=colores, edgecolor='white', height=0.75)

ax.axvline(df_2022['porcentaje'].mean(), color='#111827',
           linestyle='--', linewidth=1, label=f'Promedio nacional: {df_2022["porcentaje"].mean():.1f}%')
ax.set_xlabel('Tasa de pobreza (%)')
ax.set_title('Poblacion en situacion de pobreza por estado\nMexico, 2022',
             fontweight='bold', pad=10)
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('lab2_pobreza_2022.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Estado con mas pobreza:  {df_2022.iloc[-1].estado_corto} ({df_2022.iloc[-1].porcentaje:.1f}%)')
print(f'Estado con menos pobreza: {df_2022.iloc[0].estado_corto} ({df_2022.iloc[0].porcentaje:.1f}%)')
print(f'Razon maxima/minima: {df_2022.iloc[-1].porcentaje / df_2022.iloc[0].porcentaje:.1f}x')

### 2.4 — El ejercicio central: ¿que NO puede responder este dataset?

Esta es la parte mas importante del lab. Un analista honesto documenta los limites
de su fuente **antes** de presentar resultados — no despues de que alguien los cuestiona.

Discute con la persona a tu lado y completa la siguiente celda.

In [ ]:
evaluar_fuente(
    nombre='CONEVAL — Medicion pobreza por entidad federativa 2022',

    preguntas_posibles=[
        'Que estados tienen mas pobreza en 2022',
        'Como cambio la pobreza entre 2016 y 2022 por estado',
        'Que carencias son mas frecuentes en cada entidad',
        # Agrega las que se te ocurran
    ],

    preguntas_imposibles=[
        'Que municipios especificos concentran mas pobreza',
        'Que causo que la pobreza subiera o bajara',
        'Como estaban los estados en 2019 o 2021',
        # Agrega las que se te ocurran — al menos 3 mas
        # Pista: piensa en unidades de analisis, causalidad, y poblaciones
    ],

    poblaciones_ausentes=[
        'Personas sin domicilio fijo',
        # Agrega al menos dos mas especificas para el contexto mexicano
    ],

    notas_licencia='Libre para investigacion. Citar: CONEVAL, con base en ENIGH-INEGI 2022.'
)

### 2.5 — La pregunta mas dificil: ¿a quien le falta?

Mirar quien **no** aparece en los datos es tan importante como analizar a quien si aparece.
Este dataset mide a la poblacion que vive en viviendas particulares y fue captada
por la ENIGH. Hay grupos que esa encuesta no captura bien.

In [ ]:
# ¿A que porcentaje de la poblacion de Mexico le llegaria en teoria este dataset?
# ENIGH 2022 tiene cobertura de aproximadamente el 95% de la poblacion en viviendas
# particulares. Veamos que dice el propio dataset sobre el total de personas.

# Cargamos la hoja nacional para ver los totales
raw_nacional = pd.read_excel(ARCHIVO, sheet_name='Estados Unidos Mexicanos', header=None)

# Fila 8 = pobreza total, columnas 8-11 = miles de personas en 2016, 2018, 2020, 2022
fila_pobreza = raw_nacional.iloc[8]
print('Personas en situacion de pobreza (miles):')
for i, anio in enumerate([2016, 2018, 2020, 2022]):
    miles = fila_pobreza.iloc[8 + i]
    print(f'  {anio}: {miles:,.0f} miles = {miles/1000:.1f} millones')

print()
print('Reflexion:')
print('El Censo 2020 conto ~126 millones de personas en Mexico.')
print('¿La suma de personas en todas las categorias de pobreza llega a ese numero?')
print('Si no, ¿donde estan los que faltan?')

**Grupos que la ENIGH no captura bien — completa la lista:**

| Grupo | Por que no aparece | Politica afectada |
|---|---|---|
| Personas sin domicilio fijo | La encuesta es domiciliaria | Programas de vivienda |
| ... | ... | ... |
| ... | ... | ... |
| ... | ... | ... |

*Piensa en: trabajadores agricolas migrantes, personas en instituciones, comunidades muy pequenas.*

---
## Parte 3 — Tu proyecto: evalua tu propia fuente de datos

Aplica los mismos pasos al dataset que planeas usar para tu proyecto.
Si todavia no tienes uno, usa el de **Our World in Data — CO2**
(lo usaremos en el proyecto del curso a partir de la Semana 3):

```python
mi_df = pd.read_csv(
    'https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv'
)
```

In [ ]:
# 1. Carga tu dataset
# mi_df = pd.read_csv('URL_O_RUTA')

# 2. Primera mirada — copia y ejecuta estas tres lineas
# print(mi_df.shape)
# print(mi_df.dtypes)
# mi_df.head()

# 3. ¿Cuantos faltantes?
# mi_df.isnull().sum()

# 4. Una variable importante — ¿como se distribuye?
# mi_df['NOMBRE_VARIABLE'].describe()

# 5. Una grafica simple de esa variable
# mi_df['NOMBRE_VARIABLE'].hist(bins=30, color='#0891B2', edgecolor='white')
# plt.title('Distribucion de ...')
# plt.show()

In [ ]:
# 6. Llena esta tarjeta con tu fuente
# evaluar_fuente(
#     nombre='...',
#     preguntas_posibles=['...', '...'],
#     preguntas_imposibles=['...', '...', '...'],
#     poblaciones_ausentes=['...', '...'],
#     notas_licencia='...'
# )

---
## Para llevar

Antes de usar cualquier dataset en un analisis de politica publica,
hazte estas preguntas:

1. **¿Quien lo recolecto y con que objetivo?**
   Los datos del IMSS existen porque hay un sistema de seguridad social, no porque
   alguien quiso estudiar el trabajo informal. Eso condiciona todo lo que contienen.

2. **¿A quien incluye y a quien excluye?**
   No es una pregunta tecnica — es una pregunta politica. Las personas que no aparecen
   en los datos raramente lo hacen por azar.

3. **¿Que puedes afirmar y que no?**
   Un dataset estatal no habla de municipios. Un corte transversal no habla de causalidad.
   Saber donde termina tu dato es tan importante como saber lo que contiene.

---
**Tarea para la proxima semana:**
- Traer a clase un caso real de sesgo algoritmico o mal uso de datos (2-3 min)
- Tener identificado el dataset para tu proyecto propio
- Opcional: completar la Parte 3 con tu dataset real